In [ ]:
#Import Libraries
!pip install ortools

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [ ]:
#Import the file utility from Google Colab
from google.colab import files
# Upload the Excel File from PC
uploaded = files.upload()

Saving Statistics of Refuse Deposited at Various Landfill Sites 2020 - 2024.xlsx to Statistics of Refuse Deposited at Various Landfill Sites 2020 - 2024.xlsx


In [ ]:
#Import Pandas for data manipulation
import pandas as pd
# Read the uploaded Excel file
raw = pd.read_excel(
    "Statistics of Refuse Deposited at Various Landfill Sites 2020 - 2024.xlsx"
)

In [ ]:
#Define the row where each year's data starts
year_rows = {
    2020: 1,
    2021: 20,
    2022: 39,
    2023: 57,
    2024: 75
}

# Create an empty list to store cleaned records
all_records = []

# Loop through each year selection
for year, start_row in year_rows.items():
  #Extract 12 months of data for that year
  year_data = raw.iloc[start_row:start_row+12].reset_index(drop=True)

  #Loop through each month
  for _, row in year_data.iterrows():
    #Extract month name
    month = row.iloc[0]

    # Define column positions for each landfill
    landfill_columns = {
        "Olusosun": (1,2),
        "Abule-Egba": (3,4),
        "Solous": (5,6),
        "Badagry": (7,8),
        "Epe": (9,10),
        "Ikorodu": (11,12)
    }

    # Extract trips and tonnage for each landfill
    for landfill, (trip_col, ton_col) in landfill_columns.items():

        trips = row.iloc[trip_col]
        tonnage = row.iloc[ton_col]

        # Store cleaned record
        all_records.append({
            "Year": year,
            "Month": month,
            "Landfill": landfill,
            "Trips": trips,
            "Tonnage": tonnage
        })

# Convert list into dataframe
df = pd.DataFrame(all_records)

# Remove missing values
df.dropna(inplace=True)

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Display first few rows
df.head()

,Year,Month,Landfill,Trips,Tonnage
12,2020,JANUARY,Olusosun,10627,107577
13,2020,JANUARY,Abule-Egba,721,6489
14,2020,JANUARY,Solous,1165,10750
15,2020,JANUARY,Badagry,191,1719
16,2020,JANUARY,Epe,1393,15516


In [ ]:
import numpy as np
# Create dictionary to convert month names to numbers
month_map = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

# Convert month names into numeric values
df["Month_Number"] = df["Month"].str.title().map(month_map)

# Create quarter feature
df["Quarter"] = ((df["Month_Number"] - 1) // 3) + 1

# Create festive period indicator
# December and January are considered festive months
df["Festive_Period"] = np.where(
    df["Month"].isin(["December", "January"]),
    1,
    0
)

# Assign population density values to landfill locations
population_density = {
    "Olusosun": 15000,
    "Abule-Egba": 12000,
    "Solous": 13000,
    "Badagry": 8000,
    "Epe": 7000,
    "Ikorodu": 11000
}

# Map density values into dataset
df["Population_Density"] = df["Landfill"].map(
    population_density
)

# Encode landfill names into numeric values
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Landfill_Encoded"] = encoder.fit_transform(
    df["Landfill"]
)

# Display dataset information
print(df.head())

# Save cleaned dataset
df.to_csv(
    "cleaned_waste_dataset.csv",
    index=False
)

print("Dataset saved successfully.")

    Year    Month    Landfill  Trips Tonnage  Month_Number  Quarter  \
12  2020  JANUARY    Olusosun  10627  107577           1.0      1.0   
13  2020  JANUARY  Abule-Egba    721    6489           1.0      1.0   
14  2020  JANUARY      Solous   1165   10750           1.0      1.0   
15  2020  JANUARY     Badagry    191    1719           1.0      1.0   
16  2020  JANUARY         Epe   1393   15516           1.0      1.0   

    Festive_Period  Population_Density  Landfill_Encoded  
12               0               15000                 4  
13               0               12000                 0  
14               0               13000                 5  
15               0                8000                 1  
16               0                7000                 2  
Dataset saved successfully.


In [ ]:
# Import machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# Import evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Import joblib for model saving
import joblib

# Ensure 'Trips' and 'Tonnage' columns are numeric
# Coerce errors will turn non-numeric values into NaN
df["Trips"] = pd.to_numeric(df["Trips"], errors="coerce")
df["Tonnage"] = pd.to_numeric(df["Tonnage"], errors="coerce")

# Drop rows where 'Trips' or 'Tonnage' might have become NaN
# or if they had NaNs initially that were not caught by a previous dropna
df.dropna(subset=["Trips", "Tonnage"], inplace=True)

# Select input features
features = [
    "Year",
    "Month_Number",
    "Quarter",
    "Festive_Period",
    "Population_Density",
    "Landfill_Encoded",
    "Trips"
]

# Define feature matrix
X = df[features]

# Define target variable
y = df["Tonnage"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create Random Forest model
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train model
model.fit(X_train, y_train)

# Generate predictions
predictions = model.predict(X_test)

print("Model trained successfully.")

Model trained successfully.


In [ ]:
# Evaluate model performance
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Absolute Error (MAE): 4125.61
Mean Squared Error (MSE): 114116010.33
R-squared (R2): 0.97


In [ ]:
# Save trained model to disk
joblib.dump(
    model,
    "random_forest.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [ ]:
# Create a sample input for testing

sample = pd.DataFrame({
    "Year": [2025],
    "Month_Number": [6],
    "Quarter": [2],
    "Festive_Period": [0],
    "Population_Density": [15000],
    "Landfill_Encoded": [4],
    "Trips": [9000]
})

# Predict waste tonnage
prediction = model.predict(sample)

# Display prediction result
print(
    "Predicted Waste Tonnage:",
    round(prediction[0], 2)
)

Predicted Waste Tonnage: 85281.03


In [ ]:
#Streamlit app


# ==========================================
# SMART WASTE OPTIMIZATION SYSTEM
# STREAMLIT APPLICATION
# ==========================================

# Import required libraries
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# ==========================================
# PAGE CONFIGURATION
# ==========================================

st.set_page_config(
    page_title="Smart Waste Optimization",
    layout="wide"
)

# ==========================================%
# LOAD TRAINED MODEL
# ==========================================%

# Make sure the model file 'random_forest.pkl' is available in the Colab environment
# (It should have been saved in a previous step.)
model = joblib.load("random_forest.pkl")

# ==========================================%
# APP TITLE
# ==========================================%

st.title("♻️ Smart Waste Optimization System")

st.markdown(
    """
    This system predicts waste tonnage and supports
    waste collection route optimization across Lagos landfill sites.
    """
)

# ==========================================%
# SIDEBAR
# ==========================================%

page = st.sidebar.selectbox(
    "Select Module",
    [
        "Waste Prediction",
        "Analytics"
    ]
)

# ==========================================%
# WASTE PREDICTION PAGE
# ==========================================%

if page == "Waste Prediction":

    st.header("Waste Volume Prediction")

    landfill = st.selectbox(
        "Landfill Site",
        [
            "Olusosun",
            "Abule-Egba",
            "Solous",
            "Badagry",
            "Epe",
            "Ikorodu"
        ]
    )

    year = st.number_input(
        "Year",
        min_value=2025,
        max_value=2035,
        value=2025
    )

    month = st.selectbox(
        "Month",
        [
            "January",
            "February",
            "March",
            "April",
            "May",
            "June",
            "July",
            "August",
            "September",
            "October",
            "November",
            "December"
        ]
    )

    trips = st.number_input(
        "Expected Trips",
        min_value=1,
        value=1000
    )

    # Month Conversion
    month_map = {
        "January":1,
        "February":2,
        "March":3,
        "April":4,
        "May":5,
        "June":6,
        "July":7,
        "August":8,
        "September":9,
        "October":10,
        "November":11,
        "December":12
    }

    month_number = month_map[month]

    quarter = ((month_number - 1)//3)+1

    festive_period = 1 if month in [
        "December",
        "January"
    ] else 0

    # Population Density Mapping

    density = {
        "Olusosun":15000,
        "Abule-Egba":12000,
        "Solous":13000,
        "Badagry":8000,
        "Epe":7000,
        "Ikorodu":11000
    }

    landfill_encoding = {
        "Abule-Egba":0,
        "Badagry":1,
        "Epe":2,
        "Ikorodu":3,
        "Olusosun":4,
        "Solous":5
    }

    population_density = density[landfill]

    landfill_encoded = landfill_encoding[
        landfill
    ]

    if st.button("Predict Waste Tonnage"):

        input_data = pd.DataFrame({

            "Year":[year],

            "Month_Number":[month_number],

            "Quarter":[quarter],

            "Festive_Period":[festive_period],

            "Population_Density":[population_density],

            "Landfill_Encoded":[landfill_encoded],

            "Trips":[trips]

        })

        prediction = model.predict(
            input_data
        )[0]

        st.success(
            f"Predicted Waste Tonnage: {prediction:,.2f} tonnes"
        )

# ==========================================%
# ANALYTICS PAGE
# ==========================================%

if page == "Analytics":

    st.header("Project Analytics")

    st.info(
        """
        Model Performance Metrics obtained
        during Random Forest Evaluation.
        """
    )

    # Display actual model evaluation metrics
    st.metric(
        "MAE",
        f"{mae:.2f}" # Use the calculated MAE from previous step
    )

    # RMSE is sqrt(MSE)
    rmse = np.sqrt(mse) # Calculate RMSE
    st.metric(
        "RMSE",
        f"{rmse:.2f}"
    )

    st.metric(
        "R² Score",
        f"{r2:.2f}" # Use the calculated R2 from previous step
    )

    st.subheader(
        "Impact Analysis"
    )

    traditional_distance = 150
    optimized_distance = 95

    distance_saved = (
        traditional_distance -
        optimized_distance
    )

    fuel_saved = (
        distance_saved * 0.15
    )

    st.write(
        f"Distance Saved: {distance_saved} km"
    )

    st.write(
        f"Estimated Fuel Saved: {fuel_saved:.2f} Litres"
    )


2026-06-02 11:58:47.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.652 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.655 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.658 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.661 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.663 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.665 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 11:58:47.669 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
from google.colab import files

files.download("random_forest.pkl")
files.download("cleaned_waste_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>